In [1]:
%pip install multiprocess

In [4]:
from collections import defaultdict
from multiprocess import Pool, cpu_count
import torch
from typing import Callable, List, Tuple, Any

class MapReduce:
    def __init__(self, mapper: Callable[[Any], List[Tuple[Any, Any]]],
                 reducer: Callable[[Any, List[Any]], Any]):
        """
        Initialize with custom mapper and reducer functions.

        :param mapper: A function that maps input to key-value pairs.
        :param reducer: A function that reduces key-value pairs to output.
        """
        self.mapper = mapper
        self.reducer = reducer

    def _map_worker(self, data_chunk):
        """Process data chunk through the mapper."""
        return self.mapper(data_chunk)

    def _reduce_worker(self, kv_pairs):
        """Process key-value pairs through the reducer."""
        key, values = kv_pairs
        return self.reducer(key, values)

    def execute(self, data: List[Any], num_workers: int = None) -> List[Tuple[Any, Any]]:
        """
        Execute the MapReduce operation.

        :param data: Input data to be processed.
        :param num_workers: Number of workers for parallel processing (defaults to number of CPU cores).
        :return: The final reduced result as a list of key-value pairs.
        """
        # Map Phase: Apply mapper function to the input data in parallel.
        with Pool(num_workers) as pool:
            mapped = pool.map(self._map_worker, data)

        # Combine all mapped results into a list of key-value pairs.
        kv_store = defaultdict(list)
        for sublist in mapped:
            for key, value in sublist:
                kv_store[key].append(value)

        # Reduce Phase: Apply reducer function to each key's values.
        with Pool(num_workers) as pool:
            reduced = pool.map(self._reduce_worker, kv_store.items())

        return reduced

# Example: Word Count MapReduce

def word_count_mapper(document: str) -> List[Tuple[str, int]]:
    """
    Mapper function that splits a document into words and emits key-value pairs of (word, 1).
    """
    words = document.split()
    return [(word, 1) for word in words]

def word_count_reducer(word: str, counts: List[int]) -> Tuple[str, int]:
    """
    Reducer function that sums the occurrences of each word.
    """
    return word, sum(counts)

import re
def remover_stopwords(textos: List[str]) -> List[str]:
    STOPWORDS = {
      "à", "a", "o", "as", "os", "de", "do", "da", "dos", "das", "em", "no", "na",
      "nos", "nas", "por", "foi", "pelo", "pela", "pelos", "pelas", "com", "um",
      "uma", "uns", "umas", "para", "que", "e", "é", "ou", "se", "como", "mas"
    }

    resultado = []
    for texto in textos:
        # 1. Separa as palavras mantendo letras acentuadas/caracteres válidos
        palavras = re.findall(r'\b\w+\b', texto.lower())

        # 2. Filtra removendo as stopwords
        filtradas = [p for p in palavras if p not in STOPWORDS]

        # 3. Reagrupa as palavras filtradas em uma única string mantendo o espaço
        resultado.append(" ".join(filtradas))
    return resultado




In [3]:
if __name__ == "__main__":
    # Sample Data: List of documents (strings)
    documents = [
        "a eaj pertence à estrutura administrativa da ufrn. A eaj é mais antiga que a ufrn",
        "o curso tads foi criado em 2012, sendo a primeira turma do tads em 2013.2",
        "o tads é um curso executado na eaj, presencial, com 6 períodos",
        "venha desenvolver sistemas no tads ufrn"
    ]

    documents = remover_stopwords(documents)

    print(documents)
    # Initialize MapReduce with the custom mapper and reducer.
    map_reduce = MapReduce(mapper=word_count_mapper, reducer=word_count_reducer)

    # Execute MapReduce on the documents.
    result = map_reduce.execute(data=documents, num_workers=4)

    # Display the results
    for word, count in result:
        print(f"{word}: {count}")

['eaj pertence estrutura administrativa ufrn eaj mais antiga ufrn', 'curso tads criado 2012 sendo primeira turma tads 2013 2', 'tads curso executado eaj presencial 6 períodos', 'venha desenvolver sistemas tads ufrn']
num_workers: 4
eaj: 3
pertence: 1
estrutura: 1
administrativa: 1
ufrn: 3
mais: 1
antiga: 1
curso: 2
tads: 4
criado: 1
2012: 1
sendo: 1
primeira: 1
turma: 1
2013: 1
2: 1
executado: 1
presencial: 1
6: 1
períodos: 1
venha: 1
desenvolver: 1
sistemas: 1
